In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import glob

In [5]:
# ==========================================
# CẤU HÌNH ĐƯỜNG DẪN
# ==========================================
# Đường dẫn đến thư mục chứa file parquet features
PARQUET_DIR = r"C:\1_Workspace\4_Nam_4\1_HocKy_251\1_Do_An_Chuyen_Nganh\Urban-Traffic-Links\data\processed\parquet\traffic_features"

def find_latest_file(directory):
    """Tìm file parquet mới nhất trong thư mục"""
    # Tìm tất cả file .parquet (kể cả trong subfolder nếu có partition)
    files = glob.glob(os.path.join(directory, "**/*.parquet"), recursive=True)
    if not files:
        return None
    # Sắp xếp theo thời gian sửa đổi mới nhất
    latest_file = min(files, key=os.path.getmtime)
    return latest_file

def inspect_parquet(file_path):
    print("=" * 60)
    print(f"🔍 ĐANG KIỂM TRA FILE: {os.path.basename(file_path)}")
    print(f"📁 Đường dẫn đầy đủ: {file_path}")
    print("=" * 60)

    try:
        # Đọc file
        df = pd.read_parquet(file_path)
    except Exception as e:
        print(f"❌ Lỗi khi đọc file: {e}")
        return

    # 1. TỔNG QUAN
    print("\n[1] TỔNG QUAN DỮ LIỆU")
    print(f"- Số dòng (Rows):    {df.shape[0]:,}")
    print(f"- Số cột (Columns):  {df.shape[1]}")
    mem_usage = df.memory_usage(deep=True).sum() / 1024 / 1024
    print(f"- Dung lượng RAM:    {mem_usage:.2f} MB")
    
    # 2. KIỂM TRA CHẤT LƯỢNG (MISSING & DUPLICATE)
    print("\n[2] KIỂM TRA CHẤT LƯỢNG")
    
    # Check Null
    null_counts = df.isnull().sum()
    cols_with_nulls = null_counts[null_counts > 0]
    if not cols_with_nulls.empty:
        print("⚠️ Cảnh báo: Có giá trị NULL tại các cột:")
        print(cols_with_nulls)
    else:
        print("✅ Không có giá trị NULL.")

    # Check Duplicate Segment trong cùng 1 ngữ cảnh (nếu có cột time_set)
    if 'segment_id' in df.columns and 'time_set' in df.columns and 'date_range' in df.columns:
        # Nếu cùng 1 segment, cùng 1 ngày, cùng 1 giờ mà xuất hiện 2 lần -> Trùng lặp
        duplicates = df.duplicated(subset=['segment_id', 'time_set', 'date_range'])
        if duplicates.any():
            print(f"⚠️ Cảnh báo: Tìm thấy {duplicates.sum()} dòng bị trùng lặp (Duplicate Keys).")
        else:
            print("✅ Không có dòng trùng lặp (Unique Keys OK).")
    
    # 3. PHÂN PHỐI DỮ LIỆU (CATEGORICAL)
    print("\n[3] PHÂN PHỐI DỮ LIỆU (CATEGORICAL)")
    cat_cols = ['time_set', 'date_range', 'frc'] # Các cột phân loại quan trọng
    for col in cat_cols:
        if col in df.columns:
            print(f"\n--- Phân phối cột '{col}' ---")
            print(df[col].value_counts().head())
            print(f"(Tổng cộng {df[col].nunique()} giá trị độc nhất)")

    # 4. THỐNG KÊ SỐ HỌC (NUMERICAL)
    print("\n[4] THỐNG KÊ TỐC ĐỘ & THỜI GIAN")
    num_cols = ['average_speed', 'median_speed', 'speed_zscore', 'sample_size']
    existing_num_cols = [c for c in num_cols if c in df.columns]
    
    if existing_num_cols:
        stats = df[existing_num_cols].describe().T[['count', 'mean', 'min', '50%', 'max', 'std']]
        print(stats)
    
    # 5. XEM MẪU
    print("\n[5] 3 DÒNG DỮ LIỆU MẪU")
    pd.set_option('display.max_columns', None) # Hiện tất cả cột
    pd.set_option('display.width', 1000)
    print(df.sample(3) if len(df) > 3 else df)

if __name__ == "__main__":
    # Tự động tìm file mới nhất để check
    latest_file = find_latest_file(PARQUET_DIR)
    
    if latest_file:
        inspect_parquet(latest_file)
    else:
        print(f"❌ Không tìm thấy file .parquet nào trong thư mục: {PARQUET_DIR}")
        print("Vui lòng kiểm tra lại đường dẫn trong biến PARQUET_DIR.")

🔍 ĐANG KIỂM TRA FILE: traffic_features_20251220_164358.parquet
📁 Đường dẫn đầy đủ: C:\1_Workspace\4_Nam_4\1_HocKy_251\1_Do_An_Chuyen_Nganh\Urban-Traffic-Links\data\processed\parquet\traffic_features\traffic_features_20251220_164358.parquet

[1] TỔNG QUAN DỮ LIỆU
- Số dòng (Rows):    6,788
- Số cột (Columns):  55
- Dung lượng RAM:    4.02 MB

[2] KIỂM TRA CHẤT LƯỢNG
⚠️ Cảnh báo: Có giá trị NULL tại các cột:
street_name              108
speed_derivative        1777
speed_acceleration      3505
speed_rate_of_change    1777
speed_volatility        1777
speed_zscore              49
dtype: int64
✅ Không có dòng trùng lặp (Unique Keys OK).

[3] PHÂN PHỐI DỮ LIỆU (CATEGORICAL)

--- Phân phối cột 'time_set' ---
time_set
3.0    1769
2.0    1735
1.0    1674
0.0    1610
Name: count, dtype: int64
(Tổng cộng 4 giá trị độc nhất)

--- Phân phối cột 'date_range' ---
date_range
0.0    6788
Name: count, dtype: int64
(Tổng cộng 1 giá trị độc nhất)

--- Phân phối cột 'frc' ---
frc
3.0    3061
4.0    2887
2